In [1]:
# Load PDF 
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader 

loader = PyPDFLoader('economics_research_reference.pdf')
pages = loader.load()

In [2]:
# Building chunks 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_spliter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=140)
texts=text_spliter.split_documents(pages)
chunks = [i.page_content for i in texts]
metadata = [i.metadata for i in texts]

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding_function = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

client = chromadb.PersistentClient(path="./Database_VD")
collection = client.get_or_create_collection(name="clean_vectorDB",embedding_function=embedding_function)

if collection.count()==0:
    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))],
        metadatas=metadata
    )
